# Il repository come memoria: istruzioni e fonte di verità

Il codice del capitolo [«Il repository come memoria: istruzioni e fonte di verità»](https://book.paithon.it/main/Harness/il-repository-come-memoria.html), *Paithon Book*.

Le celle sono quelle del libro, nell'ordine in cui compaiono: il testo che le spiega sta nelle pagine, qui c'è solo la parte da eseguire e da rompere.

Generato da `scripts/genera-notebook.py`: le correzioni vanno fatte nelle pagine del libro, non qui.


> **Verificato il 2026-07-25** con torch 2.13.0, numpy 2.4.6, pandas 3.0.5, scikit-learn 1.9.0, transformers 5.14.1, diffusers 0.39.0, librosa 0.11.0, torch-geometric 2.8.0.post1. Tutte le celle di questo notebook sono state eseguite senza errori con quelle versioni; le librerie si muovono, e se qualcosa qui non gira piu' e' un errore del libro: [segnalalo](https://github.com/paithon-it/paithonbook/issues).


In [ ]:
# Su Colab quasi tutto c'è già; questa riga serve altrove.
%pip install -q numpy

In [ ]:
# Mostra il valore di ogni riga, come i commenti «# ->» del libro.
try:
    from IPython.core.interactiveshell import InteractiveShell
    InteractiveShell.ast_node_interactivity = 'all'
except ImportError:      # fuori da IPython non serve e non c'è
    pass

## Il repository come memoria: istruzioni e fonte di verità

[Leggi la pagina](https://book.paithon.it/main/Harness/il-repository-come-memoria.html)


### Tenere vera la memoria


In [ ]:
import re

# Un repository finto: percorso -> contenuto. L'indice cita i documenti e i
# comandi fra apici inversi, come si fa di solito in Markdown.
repo = {
    "AGENTS.md": (
        "# Negozio online\n"
        "Avvio: `make dev`. Controllo: `make check`, poi `make lint`.\n"
        "Mai scrivere sul database di produzione.\n"
        "- dove va il codice nuovo: `docs/architettura.md`\n"
        "- prima di toccare il checkout: `docs/pagamenti.md`\n"
        "- a inizio sessione: `docs/piani/carrello.md`\n"
    ),
    "docs/architettura.md": "Tre strati: dati, servizi, interfaccia.",
    "docs/piani/carrello.md": "Fatto: aggiunta al carrello. Manca: sconti.",
    "docs/sicurezza.md": "Le chiavi dei pagamenti non vanno mai nei file.",
    "Makefile": "dev:\n\tpython app.py\ncheck:\n\tpytest\n",
}

indice = repo["AGENTS.md"]
citati = set(re.findall(r"`(docs/[^`]+)`", indice))
documenti = {p for p in repo if p.startswith("docs/")}
comandi = set(re.findall(r"`make (\w+)`", indice))
bersagli = set(re.findall(r"^(\w+):", repo["Makefile"], re.M))

problemi = {
    "documenti citati ma assenti": sorted(citati - set(repo)),
    "documenti mai citati": sorted(documenti - citati),
    "comandi citati ma assenti": sorted(comandi - bersagli),
}
print("righe dell'indice:", len(indice.splitlines()))
for nome, elenco in problemi.items():
    print(f"{nome}: {elenco}")
print("cancello:", "chiuso" if any(problemi.values()) else "aperto")

## Il primo turno: preparare prima di costruire

[Leggi la pagina](https://book.paithon.it/main/Harness/il-primo-turno.html)


### Un cancello per l'elenco


In [ ]:
import copy

# L'elenco com'era nell'ultimo salvataggio: il ciclo lo rilegge dalla storia
# di git, non dal file che la sessione ha avuto in mano.
elenco = [
    {"category": "funzionale",
     "description": "Un utente aggiunge un prodotto e vede il totale",
     "steps": ["Apri un prodotto", "Premi «Aggiungi»", "Controlla il totale"],
     "passes": False},
    {"category": "funzionale",
     "description": "Un utente paga con la carta",
     "steps": ["Apri il carrello", "Inserisci la carta", "Ricevi la conferma"],
     "passes": False},
    {"category": "interfaccia",
     "description": "La pagina del carrello si legge su un telefono",
     "steps": ["Apri il carrello a 390 pixel", "Nessun testo esce dallo schermo"],
     "passes": False},
]

def controlla(prima, dopo):
    """Rifiuta ogni cambiamento che non sia il valore vero/falso di `passes`."""
    errori = []
    nomi_prima = [v["description"] for v in prima]
    nomi_dopo = [v["description"] for v in dopo]
    if len(set(nomi_dopo)) != len(nomi_dopo):
        errori.append("due voci con la stessa descrizione")
    for nome in sorted(set(nomi_prima) - set(nomi_dopo)):
        errori.append(f"voce tolta: {nome}")
    for nome in sorted(set(nomi_dopo) - set(nomi_prima)):
        errori.append(f"voce nuova o descrizione riscritta: {nome}")
    comuni = set(nomi_prima) & set(nomi_dopo)
    ordine_prima = [n for n in nomi_prima if n in comuni]
    ordine_dopo = [n for n in nomi_dopo if n in comuni]
    if ordine_prima != ordine_dopo:
        errori.append("voci riordinate")
    vecchie = {v["description"]: v for v in prima}
    nuove = {v["description"]: v for v in dopo}
    for nome in sorted(comuni):
        a, b = vecchie[nome], nuove[nome]
        if a.keys() != b.keys():
            errori.append(f"campi aggiunti o tolti in: {nome}")
            continue
        for campo in sorted(a):
            if campo != "passes" and a[campo] != b[campo]:
                errori.append(f"campo «{campo}» cambiato in: {nome}")
        if not isinstance(b["passes"], bool):
            errori.append(f"`passes` non è vero o falso in: {nome}")
    return errori

def avanzamento(voci):
    """Il contatore ingenuo: conta sull'elenco com'è adesso."""
    return f"{sum(v['passes'] is True for v in voci)}/{len(voci)}"

# Sessione onesta: il carrello ora funziona, e si spunta soltanto quello.
onesta = copy.deepcopy(elenco)
onesta[0]["passes"] = True

# Sessione furba: il pagamento è difficile e sparisce; la prova del carrello
# perde il controllo del totale, e il carrello risulta fatto.
furba = copy.deepcopy(elenco)
del furba[1]
furba[0]["steps"] = ["Apri un prodotto", "Premi «Aggiungi»"]
furba[0]["passes"] = True

for nome, dopo in [("onesta", onesta), ("furba", furba)]:
    errori = controlla(elenco, dopo)
    esito = "accettata" if not errori else "RIFIUTATA"
    print(f"sessione {nome}: avanzamento {avanzamento(dopo)}, {esito}")
    for e in errori:
        print("   ", e)

## Lavorare a turni: il passaggio di consegne

[Leggi la pagina](https://book.paithon.it/main/Harness/passaggio-di-consegne.html)


### Compattare o ripartire da capo


In [ ]:
import math

# Un modello a numeri tondi, non una misura: a ogni giro il modello rilegge
# tutto il contesto, e la cronologia cresce della sua risposta e dell'esito
# di uno strumento.
FINESTRA = 100_000      # token che il modello legge al massimo in una volta
FISSO = 5_000           # istruzioni e catalogo degli strumenti, a ogni giro
GIRO = 3_000            # quanto cresce la cronologia a ogni giro
GIRI = 60               # la lunghezza del compito
RIASSUNTO = 4_000       # la cronologia compattata diventa un riassunto
REGISTRO = 2_000        # registro, voci aperte dell'elenco, storia recente
PER_SESSIONE = 15       # giri di una sessione: una funzionalità

def senza_niente():
    letti, storia, contesti = 0, 0, []
    for t in range(GIRI):
        contesto = FISSO + storia
        if contesto > FINESTRA:          # non ci sta più: il lavoro si ferma
            break
        contesti.append(contesto)
        letti += contesto
        storia += GIRO
    return letti, contesti

def con_compattazione(soglia):
    letti, storia, contesti, volte, per_riassunti = 0, 0, [], 0, 0
    for t in range(GIRI):
        if FISSO + storia > soglia * FINESTRA:
            per_riassunti += FISSO + storia   # la chiamata che lo scrive
            storia, volte = RIASSUNTO, volte + 1
        contesto = FISSO + storia
        contesti.append(contesto)
        letti += contesto
        storia += GIRO
    return letti + per_riassunti, contesti, volte, per_riassunti

def con_ripartenza(giri=GIRI, per_sessione=PER_SESSIONE):
    letti, contesti = 0, []
    for t in range(giri):
        dentro = t % per_sessione        # giri già fatti in questa sessione
        contesto = FISSO + REGISTRO + dentro * GIRO
        contesti.append(contesto)
        letti += contesto
    return letti, contesti, math.ceil(giri / per_sessione)

def it(n):                               # 1648000 -> 1.648.000
    return f"{n:,}".replace(",", ".")

a, ca = senza_niente()
b, cb, volte, _ = con_compattazione(0.8)
b5, cb5, volte5, scrivere5 = con_compattazione(0.5)
c, cc, sessioni = con_ripartenza()
meta = len(ca) // 2
print(f"senza niente: si ferma dopo {len(ca)} giri di {GIRI}, "
      f"letti {it(a)} ({it(sum(ca[:meta]))} nei primi {meta})")
print(f"compattazione all'80%: {volte} riassunti, picco {it(max(cb))}, "
      f"letti {it(b)}")
print(f"compattazione al 50%: {volte5} riassunti, picco {it(max(cb5))}, "
      f"letti {it(b5)} ({it(scrivere5)} per scriverli)")
print(f"ripartenza: {sessioni} sessioni, picco {it(max(cc))}, letti {it(c)}")
print(f"letti per giro, in media: {it(b // GIRI)}, {it(b5 // GIRI)}, "
      f"{it(c // GIRI)}")

# Due voci nella stessa sessione, contro due sessioni da una voce.
insieme = con_ripartenza(2 * PER_SESSIONE, 2 * PER_SESSIONE)[0]
separate = con_ripartenza(2 * PER_SESSIONE, PER_SESSIONE)[0]
print(f"due voci: {it(insieme)} in una sessione, {it(separate)} in due")

## Finito vuol dire collaudato: la verifica di un lavoro lungo

[Leggi la pagina](https://book.paithon.it/main/Harness/finito-vuol-dire-collaudato.html)


### Il collaudo da utente


In [ ]:
# Due componenti del negozio, scritti in due sessioni diverse.
class Carrello:
    """Tiene i prezzi in centesimi, come si fa per non perdere spiccioli."""
    def __init__(self):
        self.righe = []
    def aggiungi(self, nome, prezzo_euro):
        self.righe.append((nome, round(prezzo_euro * 100)))
    def totale(self):
        return sum(centesimi for _, centesimi in self.righe)

def pagina_totale(carrello):
    """La pagina del carrello: chi l'ha scritta credeva il totale in euro."""
    return f"Totale: {carrello.totale():.2f} €".replace(".", ",")

# I test d'unità: ciascuno prova il suo pezzo da solo.
def test_carrello():
    c = Carrello()
    c.aggiungi("tazza", 19.99)
    return c.totale() == 1999                     # centesimi: giusto

class CarrelloFinto:                              # la controfigura del test
    def totale(self):                             # della pagina: risponde in
        return 19.99                              # euro, come la pagina crede

def test_pagina():
    return pagina_totale(CarrelloFinto()) == "Totale: 19,99 €"

# Il collaudo della voce: i pezzi veri, montati insieme, e i suoi passi di
# prova. Qui si legge lo schermo come una stringa; un harness vero passa da
# un browser.
def collaudo_carrello():
    c = Carrello()
    c.aggiungi("tazza", 19.99)                    # «aggiungi una tazza»
    schermo = pagina_totale(c)
    return schermo == "Totale: 19,99 €", schermo  # «controlla il totale»

voce = {"description": "Un utente aggiunge un prodotto e vede il totale",
        "steps": ["Aggiungi una tazza da 19,99 €",
                  "Controlla che il totale sia il prezzo del prodotto"],
        "passes": False}
collaudi = {voce["description"]: collaudo_carrello}   # un collaudo per voce

def chiudi_voce(voce, dichiarazione_agente):
    """Il campo `passes` lo scrive il collaudo della voce; la dichiarazione
    dell'agente non entra nel conto."""
    ok, schermo = collaudi[voce["description"]]()
    voce["passes"] = ok
    return ok, schermo

print("test d'unità del carrello:", "passa" if test_carrello() else "fallisce")
print("test d'unità della pagina:", "passa" if test_pagina() else "fallisce")
ok, schermo = chiudi_voce(voce, dichiarazione_agente="fatto, i test passano")
print("collaudo da utente:", "passa" if ok else "fallisce",
      f"(lo schermo dice «{schermo}»)")
print("passes:", voce["passes"])

## Gli strumenti: l'interfaccia fra l'agente e il computer

[Leggi la pagina](https://book.paithon.it/main/Harness/gli-strumenti.html)


### Che cosa torna indietro


In [ ]:
import textwrap

# Un log di 4.000 righe con un solo errore, alla riga 2.731: a numeri tondi,
# non una misura.
log = [f"{i:04d} INFO richiesta servita" for i in range(1, 4001)]
log[2730] = "2731 ERROR connessione al database rifiutata"
TETTO = 100                            # righe che lo strumento restituisce

def it(n):                             # 200000 -> 200.000
    return f"{n:,}".replace(",", ".")

def prime_righe(righe):
    """Il comando lanciato così com'è, tagliato al tetto."""
    return righe[:TETTO]

def progettato(righe):
    """Al più TETTO righe in tutto: prima gli errori, con un tetto anche
    loro, poi testa e coda del log, e una riga su come chiedere il resto."""
    if len(righe) <= TETTO:                       # un log corto passa intero
        return list(righe)
    errori = [i for i, r in enumerate(righe) if " ERROR " in r]
    cima = errori[:TETTO // 2]
    posto = TETTO - len(cima) - 1
    testa = [i for i in range(posto // 2) if i not in cima]
    coda = [i for i in range(len(righe) - (posto - posto // 2), len(righe))
            if i not in cima]
    nota = (f"[omesse {it(len(righe) - len(cima + testa + coda))} righe; "
            f"errori: {it(len(errori))}, in cima: {len(cima)}; per una "
            f"fetta: fetta_log(da, a); per cercare: cerca_log(testo)]")
    return ([righe[i] for i in cima + testa] + [nota]
            + [righe[i] for i in coda])

PASSI, ARRIVO = 60, 10                 # il log arriva al passo 10 di 60
for nome, uscita in [("tutto il log", log),
                     ("prime cento righe", prime_righe(log)),
                     ("progettato", progettato(log))]:
    visto = "sì" if any(" ERROR " in r for r in uscita) else "no"
    riletture = len(uscita) * (PASSI - ARRIVO)    # dal passo 11 al 60
    print(f"{nome:>17}: {it(len(uscita)):>5} righe, errore in vista: "
          f"{visto}, righe rilette: {it(riletture)}")
uscita = progettato(log)
print(uscita[0])
print(textwrap.fill(next(r for r in uscita if r.startswith("[omesse")),
                    76, subsequent_indent=" "))
pieno = [r.replace("INFO", "ERROR") for r in log]   # e se fosse tutto errori?
print(f"con {it(len(pieno))} errori: {len(progettato(pieno))} righe")

### Trovare il codice e modificarlo


In [ ]:
import difflib

# Un file di 120 righe: ventiquattro funzioni da cinque righe. La modifica
# rinomina il parametro di una sola funzione, e cambia 4 righe.
def funzione(i, nome):
    return [f"def tassa_{i}({nome}):",
            f"    if {nome} <= 0:",
            f"        raise ValueError({nome})",
            f"    return {nome} * {i + 2} // 100",
            ""]

vecchio = [r for i in range(24) for r in funzione(i, "importo")]
nuovo = [r for i in range(24)
         for r in funzione(i, "centesimi" if i == 10 else "importo")]
cambiate = [r for r in range(len(vecchio)) if vecchio[r] != nuovo[r]]

def ancora(righe, da, a):
    """La finestra più stretta attorno alle righe da..a che compare una
    volta sola nel file: è l'ancora minima di un cerca e sostituisci."""
    for larga in range(a - da + 1, len(righe) + 1):
        for inizio in range(max(0, a + 1 - larga), da + 1):
            pezzo = righe[inizio:inizio + larga]
            if sum(righe[j:j + larga] == pezzo
                   for j in range(len(righe) - larga + 1)) == 1:
                return larga
    return len(righe)

diff = list(difflib.unified_diff(vecchio, nuovo, "a/tasse.py", "b/tasse.py",
                                 lineterm="", n=3))
larga = ancora(vecchio, cambiate[0], cambiate[-1])   # un blocco contiguo
righe_scritte = {"riscrittura intera": len(nuovo),
                 "diff unificato": len(diff),
                 "cerca e sostituisci": 2 * larga,   # ancora e sostituto
                 "operazione sull'albero": 1}        # rinomina(tassa_10, ...)
print(f"righe del file: {len(vecchio)}, righe cambiate: {len(cambiate)}")
riga = "    if importo <= 0:"
print(f"{riga.strip()!r} compare {vecchio.count(riga)} volte; "
      f"l'ancora minima è di {larga} righe")
for formato, n in righe_scritte.items():
    print(f"{formato:>22}: {n:>3} {'riga' if n == 1 else 'righe'} "
          f"da scrivere")

In [ ]:
import hashlib
import textwrap

# Lo stesso file di 120 righe, in un archivio finto in memoria, e due
# sessioni, A e B, che ci lavorano insieme.
def funzione(i, nome="importo"):
    return [f"def tassa_{i}({nome}):", f"    if {nome} <= 0:",
            f"        raise ValueError({nome})",
            f"    return {nome} * {i + 2} // 100", ""]

archivio = {"tasse.py": "\n".join(r for i in range(24) for r in funzione(i))}
letture = {}            # (sessione, percorso) -> impronta dell'ultima lettura

def impronta(testo):
    return hashlib.sha256(testo.encode("utf-8")).hexdigest()

def leggi(sessione, percorso):
    letture[sessione, percorso] = impronta(archivio[percorso])
    return archivio[percorso]

def righe_di(testo, ancora):
    """Le righe (da 1) in cui comincia ogni occorrenza dell'ancora."""
    trovate, pos = [], testo.find(ancora)
    while pos != -1:
        trovate.append(testo.count("\n", 0, pos) + 1)
        pos = testo.find(ancora, pos + 1)
    return trovate

def modifica(sessione, percorso, ancora, sostituto):
    testo = archivio[percorso]
    if (sessione, percorso) not in letture:
        return f"rifiutata: non hai mai letto {percorso}; leggilo prima"
    if letture[sessione, percorso] != impronta(testo):
        return (f"rifiutata: {percorso} è cambiato dopo la tua ultima "
                f"lettura; rileggilo e rifai la modifica sul testo nuovo")
    dove = righe_di(testo, ancora)
    if not dove:
        return ("rifiutata: l'ancora non compare nel file; copiala dal "
                "testo letto, spazi e rientri compresi")
    if len(dove) > 1:
        elenco = ", ".join(map(str, dove[:5])) + (", ..." if len(dove) > 5
                                                  else "")
        return (f"rifiutata: l'ancora compare {len(dove)} volte (righe "
                f"{elenco}); allargala con una riga vicina che la renda "
                f"unica")
    archivio[percorso] = testo.replace(ancora, sostituto, 1)
    # La sessione conosce la propria modifica: la si registra come letta,
    # così può farne un'altra senza rileggere (ma vedi il ritentativo).
    letture[sessione, percorso] = impronta(archivio[percorso])
    fine = dove[0] + ancora.count("\n")
    if fine == dove[0]:
        return f"fatta: riga {fine} sostituita"
    return f"fatta: righe {dove[0]}-{fine} sostituite"

def mostra(sessione, esito):                       # a capo a 76 colonne
    print(textwrap.fill(f"{sessione}: {esito}", 76, subsequent_indent="   "))

blocco = "\n".join(funzione(10)[:4])               # la funzione tassa_10
nuovo = "\n".join(funzione(10, "centesimi")[:4])

leggi("A", "tasse.py")
mostra("A", modifica("A", "tasse.py", "    if importo <= 0:",
                     "    if centesimi <= 0:"))
mostra("A", modifica("A", "tasse.py", "def tassa_10(importo) :", nuovo))
leggi("B", "tasse.py")                             # l'altra sessione
mostra("B", modifica("B", "tasse.py", "def tassa_23(importo):",
                     "def tassa_23(importo, aliquota=1):"))
mostra("A", modifica("A", "tasse.py", blocco, nuovo))
leggi("A", "tasse.py")
mostra("A", modifica("A", "tasse.py", blocco, nuovo))
mostra("A", modifica("A", "tasse.py", blocco, nuovo))   # l'esito si era perso

## Quando nessuno guarda: osservabilità ed entropia

[Leggi la pagina](https://book.paithon.it/main/Harness/quando-nessuno-guarda.html)


### Seguire una richiesta da capo a fondo


In [ ]:
import itertools
import json

contatore = itertools.count(1)
eventi = []                                   # il giornale degli eventi

def registra(richiesta, fase, **campi):
    """Un evento per fase, sempre con l'identificativo della richiesta."""
    eventi.append({"richiesta": richiesta, "fase": fase, **campi})

# L'archivio dei documenti, codificati mesi fa in vettori da 8 numeri.
archivio = {f"doc{i}": [1.0] * 8 for i in range(1000)}

def codifica(testo):
    """Il codificatore, aggiornato ieri: ora produce vettori da 4."""
    return [0.5] * 4

def cerca(richiesta, domanda):
    q = codifica(domanda)
    registra(richiesta, "codifica", lunghezza=len(q))
    candidati, scartate = [], set()           # lunghezze dei vettori scartati
    for nome, v in archivio.items():
        if len(v) != len(q):                  # «prudente»: salta in silenzio
            scartate.add(len(v))
            continue
        candidati.append(nome)
    registra(richiesta, "recupero", visti=len(archivio),
             compatibili=len(candidati), scartate=sorted(scartate))
    return candidati

def rispondi(domanda):
    richiesta = f"r{next(contatore)}"         # lega fra loro gli eventi
    documenti = cerca(richiesta, domanda)
    registra(richiesta, "risposta", documenti=len(documenti))
    if not documenti:
        return "Non ho trovato documenti pertinenti."
    return f"Ecco {len(documenti)} documenti."

# Quello che vede chi usa il servizio: una risposta plausibile.
print(rispondi("Come si restituisce un prodotto difettoso?"))

# Quello che dice il giornale, seguendo la richiesta da capo a fondo.
traccia = [e for e in eventi if e["richiesta"] == "r1"]
for e in traccia:
    print(json.dumps(e, ensure_ascii=False))

dove = next(e for e in traccia if e.get("compatibili") == 0)
print(f"si azzera in «{dove['fase']}»: domanda da {traccia[0]['lunghezza']} "
      f"numeri, archivio da {dove['scartate'][0]}")

### L'entropia, e la pulizia che non finisce


In [ ]:
# Un modello a numeri tondi, non una misura: b è la frazione del codice che
# segue uno schema storto. A ogni sessione si riscrive una parte del codice,
# e il codice nuovo copia gli schemi che trova.
RISCRITTO = 0.2         # frazione del codice riscritta a ogni sessione
ERRORE = 0.1            # probabilità di sbagliare copiando uno schema sano
CONTAGIO = 0.4          # quanto più volentieri si copia una scorciatoia
ONDA = 5                # la pulizia a ondate passa ogni cinque sessioni
SESSIONI, REGIME = 600, 300          # le medie si prendono sulle ultime 300

def copia(b, contagio):
    """Probabilità di copiare uno schema storto, se lo è la frazione b."""
    return b + contagio * b * (1 - b)

def deriva(contagio, permanenza=None, a_ondate=False, sessioni=SESSIONI):
    """La frazione storta sessione per sessione. La pulizia corregge ogni
    volta una quota del codice storto, scelta perché uno schema storto resti
    in media `permanenza` sessioni prima di essere corretto."""
    if permanenza is None:
        quota = 0.0
    elif a_ondate:      # in media si aspetta (ONDA - 1) / 2 la prima ondata
        quota = ONDA / (permanenza + (ONDA - 1) / 2)
    else:
        quota = 1 / permanenza
    b, storia = 0.0, []
    for k in range(1, sessioni + 1):
        c = copia(b, contagio)
        nuovo = c + ERRORE * (1 - c)     # il codice riscritto è storto
        tolta = quota if (k % ONDA == 0 or not a_ondate) else 0.0
        b = (1 - RISCRITTO) * b + RISCRITTO * nuovo - tolta * b
        storia.append(b)
    return storia

def media(storia):
    return sum(storia[-REGIME:]) / REGIME

def it(x):                               # 0.167 -> 0,167
    return f"{x:.3f}".replace(".", ",")

print("permanenza   in proporzione   con contagio")
for p in (2, 5, 10, 20):
    print(f"{p:>2} sessioni {it(media(deriva(0, p))):>16} "
          f"{it(media(deriva(CONTAGIO, p))):>14}")
for contagio, nome in [(0, "in proporzione"), (CONTAGIO, "con contagio")]:
    onde = deriva(contagio, 10, a_ondate=True)[-REGIME:]
    print(f"a ondate, permanenza 10, {nome}: media {it(media(onde))}, "
          f"picco {it(max(onde))}")
print(f"senza pulizia, dopo {SESSIONI} sessioni: "
      f"{it(deriva(0)[-1])} e {it(deriva(CONTAGIO)[-1])}")

## Il conto di un compito: token, cache e attesa

[Leggi la pagina](https://book.paithon.it/main/Harness/il-conto.html)


### La cache del prefisso, dal lato di chi paga


In [ ]:
# Il modello a numeri tondi del passaggio di consegne, non una misura, con in
# più la cache del prefisso e un prezzo. Il contesto di una chiamata è una fila
# di pezzi (nome, token): due pezzi con lo stesso nome sono identici token per
# token. Ipotesi: la cache tiene il contesto della chiamata precedente e non
# scade fra un giro e l'altro; scriverci costa quanto leggere un token fresco;
# il registro lo aggiorna l'ultimo giro di ogni sessione, dentro la sua uscita.
FINESTRA, FISSO, GIRO, GIRI = 100_000, 5_000, 3_000, 60
RIASSUNTO, REGISTRO, PER_SESSIONE = 4_000, 2_000, 15
USCITA = 500            # dei 3.000 token di un giro, quelli scritti dal modello
ORA = 20                # una riga con l'ora corrente, se la si mette in testa
# Prezzi illustrativi, in token freschi equivalenti: i listini cambiano.
P_CACHE = 0.1           # un token letto dalla cache
P_USCITA = 5            # un token scritto dal modello

def chiamate(strategia, ora=False):
    """Le chiamate di un compito: per ciascuna il contesto e i token scritti."""
    elenco, storia = [], []
    for t in range(GIRI):
        if strategia == "ripartenza" and t % PER_SESSIONE == 0:
            storia = [(f"registro {t}", REGISTRO)]       # sessione nuova
        testa = [(f"ora {t}", ORA)] if ora else []
        contesto = testa + [("fisso", FISSO)] + storia
        if strategia == "nessuna" and sum(n for _, n in contesto) > FINESTRA:
            break                                        # non ci sta più
        soglia = {"80%": 0.8, "50%": 0.5}.get(strategia)
        if soglia and sum(n for _, n in contesto) > soglia * FINESTRA:
            elenco.append((contesto, RIASSUNTO))   # la chiamata che riassume
            storia = [(f"riassunto {t}", RIASSUNTO)]
            contesto = testa + [("fisso", FISSO)] + storia
        elenco.append((contesto, USCITA))
        storia = storia + [(f"giro {t}", GIRO)]          # risposta ed esito
    return elenco

def comune(prima, dopo):
    """I token del prefisso comune: i pezzi uguali dall'inizio, in fila."""
    h = 0
    for a, b in zip(prima, dopo):
        if a != b:
            break
        h += a[1]
    return h

def conto(elenco, p_cache=P_CACHE):
    """Token letti, di cui dalla cache e freschi, token scritti, e il costo."""
    letti = cache = scritti = 0
    prima = []
    for contesto, u in elenco:
        letti += sum(n for _, n in contesto)
        cache += comune(prima, contesto)   # il prefisso della chiamata prima
        scritti += u
        prima = contesto
    freschi = letti - cache
    costo = p_cache * cache + freschi + P_USCITA * scritti
    return letti, cache, freschi, scritti, costo

def it(x, cifre=0):                      # 1648000 -> 1.648.000, 0.07 -> 0,07
    s = f"{x:,.{cifre}f}"
    return s.replace(",", "_").replace(".", ",").replace("_", ".")

print(f"{'':25}{'letti':>10}{'in cache':>11}{'freschi':>10}{'scritti':>8}"
      f"{'costo':>10}{'senza cache':>12}")
casi = {}
for nome, s, ora in [("compattazione all'80%", "80%", False),
                     ("compattazione al 50%", "50%", False),
                     ("ripartenza", "ripartenza", False),
                     ("ripartenza, ora in testa", "ripartenza", True)]:
    v = casi[nome] = conto(chiamate(s, ora))
    senza = conto(chiamate(s, ora), p_cache=1)[4]
    print(f"{nome:25}" + "".join(f"{it(x):>{w}}" for x, w in
          zip(v, (10, 11, 10, 8, 10))) + f"{it(senza):>12}")

_, c, f, u, costo = casi["ripartenza"]
print(f"ripartenza, il costo: {it(P_CACHE * c)} dalla cache, {it(f)} freschi, "
      f"{it(P_USCITA * u)} scritti")
print(f"l'ora in testa lo moltiplica per "
      f"{it(casi['ripartenza, ora in testa'][4] / costo, 1)}")

# Sotto quale prezzo della cache compattare tardi costa meno che presto?
_, c8, f8, u8, _ = casi["compattazione all'80%"]
_, c5, f5, u5, _ = casi["compattazione al 50%"]
pari = (f5 + P_USCITA * u5 - f8 - P_USCITA * u8) / (c8 - c5)
print(f"all'80% costa meno che al 50% se la cache costa meno di {it(pari, 3)}")

# Senza strategie, fino al giro in cui il contesto non ci sta più.
el = chiamate("nessuna")
_, c, f, u, _ = conto(el)
print(f"nessuna strategia, {len(el)} giri: {it(P_CACHE * c)} dalla cache, "
      f"{it(f)} freschi, {it(P_USCITA * u)} scritti")
cresce = [conto(el, p)[4] / conto(el[:16], p)[4] for p in (P_CACHE, 1)]
print(f"{len(el)} giri costano {it(cresce[0], 1)} volte 16 con la cache, "
      f"{it(cresce[1], 1)} senza")

# Dove il termine quadratico raggiunge quelli lineari, all'ordine principale.
lineari = GIRO + P_CACHE * FISSO + P_USCITA * USCITA
print(f"il quadrato raggiunge i termini lineari a "
      f"{it(2 * lineari / (P_CACHE * GIRO))} giri con la cache, a "
      f"{it(2 * (FISSO + P_USCITA * USCITA) / GIRO)} senza")

### Ogni risparmio ha un contro-costo


In [ ]:
from math import comb

# Due harness sugli stessi cento compiti, un'esecuzione per compito, esiti
# appaiati: numeri tondi, non una misura. B affida una parte dei passi a un
# modello più economico; A li lascia tutti al modello forte.
ENTRAMBI, SOLO_A, SOLO_B, NESSUNO = 70, 12, 6, 12
COMPITI = ENTRAMBI + SOLO_A + SOLO_B + NESSUNO
SPESA = {"A": 180, "B": 114}     # su tutti i compiti, fallimenti compresi
RIUSCITI = {"A": ENTRAMBI + SOLO_A, "B": ENTRAMBI + SOLO_B}

def it(x, cifre=2):
    return f"{x:.{cifre}f}".replace(".", ",")

for h in "AB":
    s = RIUSCITI[h]
    print(f"{h}: riusciti {s} su {COMPITI}, per tentativo "
          f"{it(SPESA[h] / COMPITI)}, per successo {it(SPESA[h] / s)}")

# McNemar esatto: contano solo i compiti su cui i due esiti differiscono.
n, k = SOLO_A + SOLO_B, min(SOLO_A, SOLO_B)
p = min(1, 2 * sum(comb(n, i) for i in range(k + 1)) / 2 ** n)
print(f"discordanti: {SOLO_A} riusciti solo ad A, {SOLO_B} solo a B, "
      f"p = {it(p)}")
netti = RIUSCITI["A"] - RIUSCITI["B"]
print(f"A spende {SPESA['A'] - SPESA['B']} in più per {netti} successi netti, "
      f"{it((SPESA['A'] - SPESA['B']) / netti)} l'uno")

### Il tempo di un agente


In [ ]:
# Lo stesso lavoro in tre modi, a numeri tondi e non misurati: tre letture che
# non dipendono l'una dall'altra, poi la risposta. Il modello pensa PENSA
# secondi prima di ogni chiamata, ogni lettura dura LEGGI, il modello rapido
# che prova a indovinare la chiamata dopo ne impiega INDOVINA.
PENSA, LEGGI, INDOVINA = 4, 3, 1
LETTURE = {"A": True, "B": False, "C": True}   # previsione giusta o sbagliata

def passo(giusta):
    """Il modello pensa e chiede una lettura. Se la previsione era giusta, la
    lettura è partita appena il modello rapido ha finito di indovinarla."""
    return max(PENSA, INDOVINA + LEGGI) if giusta else PENSA + LEGGI

in_serie = len(LETTURE) * (PENSA + LEGGI) + PENSA
in_parallelo = PENSA + LEGGI + PENSA     # tre letture in un passo solo
speculativo = sum(passo(g) for g in LETTURE.values()) + PENSA
sbagliate = sum(not g for g in LETTURE.values())
print(f"in serie: {in_serie} s, {len(LETTURE) + 1} chiamate al modello")
print(f"in parallelo: {in_parallelo} s, 2 chiamate al modello")
print(f"speculativo: {speculativo} s, {len(LETTURE) + 1} chiamate al modello, "
      f"{len(LETTURE)} previsioni, {sbagliate} lettura buttata")

for q in (0.25, 0.5, 0.75, 1):
    medio = q * passo(True) + (1 - q) * passo(False)
    print(f"previsione giusta con probabilità {q}: passo medio {medio} s, "
          f"invece di {PENSA + LEGGI}".replace(".", ","))

## Dal ciclo al grafo: il disegno di un lavoro che dura

[Leggi la pagina](https://book.paithon.it/main/Harness/dal-ciclo-al-grafo.html)


### Il grafo come struttura dati


In [ ]:
# Un grafo minimo per un lavoro lungo. I nodi leggono lo stato e restituiscono
# un aggiornamento; ogni campo dello stato ha la sua regola di fusione.
FUSIONE = {
    "registro": lambda vecchio, nuovo: vecchio + nuovo,    # si accoda
    "tentativi": lambda vecchio, nuovo: vecchio + nuovo,   # si somma
    "esito": lambda vecchio, nuovo: nuovo,                 # si sostituisce
}

def fondi(stato, aggiornamento):
    nuovo = dict(stato)
    for campo, valore in aggiornamento.items():
        nuovo[campo] = FUSIONE[campo](stato[campo], valore)
    return nuovo

def pianifica(stato):
    return {"registro": ["piano: una voce, il totale del carrello"]}

def costruisci(stato):
    tentativo = stato["tentativi"] + 1
    return {"registro": [f"tentativo {tentativo} costruito"], "tentativi": 1}

guasti = {"collauda": 1}             # il primo collaudo si interrompe

def collauda(stato):
    if guasti["collauda"]:
        guasti["collauda"] -= 1
        raise RuntimeError("la macchina si è spenta")     # un guasto simulato
    tentativo = stato["tentativi"]
    esito = "passato" if tentativo >= 2 else "fallito"
    riga = f"collaudo del tentativo {tentativo}: {esito}"
    return {"esito": esito, "registro": [riga]}

def rilascia(stato):
    return {"registro": ["rilasciato ai clienti"]}

NODI = {"pianifica": pianifica, "costruisci": costruisci,
        "collauda": collauda, "rilascia": rilascia}
ARCHI = {"pianifica": "costruisci", "costruisci": "collauda", "rilascia": None}
TETTO = 3                            # tentativi al massimo, poi ci si ferma

def dopo_il_collaudo(stato):         # l'unico arco che dipende dallo stato
    if stato["esito"] == "passato":
        return "rilascia"
    return "costruisci" if stato["tentativi"] < TETTO else None

CONDIZIONALI = {"collauda": dopo_il_collaudo}
IRREVERSIBILI = {"rilascia"}         # prima di questi serve una persona

# I punti salvati, uno per filo di lavoro. Qui è un dizionario; in un harness
# vero è un file o un database, scritto prima di passare al nodo dopo.
archivio = {}

def esegui(filo, approvati=()):
    """Riprende il filo dall'ultimo punto salvato e va avanti finché può.
    `approvati` sono i nodi irreversibili che una persona ha approvato."""
    inizio = {"registro": [], "tentativi": 0, "esito": None}
    nodo, stato = archivio.get(filo, ("pianifica", inizio))
    fatti = []
    while nodo is not None:
        if nodo in IRREVERSIBILI:
            if nodo not in approvati:
                return fatti, f"fermo prima di «{nodo}»: serve un'approvazione"
            stato = fondi(stato, {"registro": [f"«{nodo}» approvato"]})
        try:
            aggiornamento = NODI[nodo](stato)
        except RuntimeError as guasto:
            return fatti, f"guasto in «{nodo}»: {guasto}"
        stato = fondi(stato, aggiornamento)
        esito = aggiornamento.get("esito")
        fatti.append(f"{nodo} ({esito})" if esito else nodo)
        if nodo in CONDIZIONALI:
            nodo = CONDIZIONALI[nodo](stato)
        else:
            nodo = ARCHI[nodo]
        archivio[filo] = (nodo, stato)        # salvato dopo ogni passo
    if stato["esito"] == "fallito":           # uscito dal ciclo per il tetto
        return fatti, f"fermo dopo {TETTO} tentativi: serve una persona"
    return fatti, "fine"

for giro, approvati in [("prima corsa", ()), ("ripresa", ()),
                        ("dopo l'approvazione", ("rilascia",))]:
    fatti, motivo = esegui("ordine-17", approvati)
    print(f"{giro}: {' -> '.join(fatti)}")
    print(f"   {motivo}")
    if giro == "prima corsa":
        nodo, stato = archivio["ordine-17"]
        print(f"   punto salvato: si riparte da «{nodo}», "
              f"tentativi costruiti: {stato['tentativi']}")
print("registro del filo:")
for riga in archivio["ordine-17"][1]["registro"]:
    print(f"   {riga}")

## L'harness che si riscrive: quando a cercarlo è un agente

[Leggi la pagina](https://book.paithon.it/main/Harness/l-harness-che-si-riscrive.html)


### Dall'esperienza al contesto


In [ ]:
import numpy as np

# Un modello a numeri tondi, non una misura. I compiti sono di 40 tipi, e per
# ogni tipo esiste una lezione che lo fa riuscire; dopo ogni compito chi
# riflette scrive la lezione di quel tipo, qui sempre giusta. Il contesto è
# l'insieme delle lezioni che il compito successivo trova scritte.
TIPI, COMPITI, PROVE = 40, 400, 200
TIENE = 0.95    # riscrittura: ogni lezione passa nella copia con questa prob.
CROLLO = 0.01   # riscrittura: probabilità che la copia butti via tutto
rng = np.random.default_rng(0)

def una_prova(regola):
    contesto, trovate, crolli = set(), [], 0
    for t in rng.integers(TIPI, size=COMPITI):
        trovate.append(t in contesto)        # il compito trova la sua lezione?
        if regola == "crolla" and rng.random() < CROLLO:
            contesto, crolli = set(), crolli + 1
        elif regola in ("riscrivi", "crolla"):
            contesto = {m for m in contesto if rng.random() < TIENE}
        contesto.add(t)                # la lezione nuova; un doppione si scarta
    return np.mean(trovate[COMPITI // 2:]), len(contesto), crolli

for regola, nome in [("riscrivi", "riscrittura intera"),
                     ("crolla", "riscrittura con i crolli"),
                     ("fondi", "voci fuse una alla volta")]:
    esiti = np.mean([una_prova(regola) for _ in range(PROVE)], axis=0)
    print(f"{nome:<25} lezione trovata {esiti[0]:.2f}, "
          f"lezioni {esiti[1]:.1f}, crolli {esiti[2]:.1f}")
q, nu = TIENE, TIPI
P = 1 / (q + nu * (1 - q))           # la formula, a regime
print(f"formula: lezione trovata {P:.2f}, lezioni presenti {nu * P:.1f}; "
      f"con una lezione nuova a ogni compito {1 / (1 - q):.0f}")

### L'harness come programma cercato


In [ ]:
import numpy as np

# Un modello a numeri tondi, non una misura. Un harness è una fila di
# componenti, accesi o spenti, e una modifica ne cambia uno a caso. Il successo
# vero s(H) è inventato, in due versioni: nella "liscia" ogni componente
# acceso aggiunge un poco; nella "trappola", costruita apposta, i primi tre
# servono solo tutti insieme, e uno o due da soli peggiorano.
GIRI, PROVE = 80, 300

def liscia(h):
    return 0.20 + 0.04 * h.sum()

def trappola(h):
    blocco = h[:3].sum()
    bonus = 0.30 if blocco == 3 else -0.06 * blocco
    return 0.20 + 0.03 * h[3:].sum() + bonus

def voto(rng, s, compiti):     # compiti=None: misura esatta; altrimenti
    if compiti is None:        # un'estrazione nuova a ogni valutazione
        return s
    return rng.binomial(compiti, s) / compiti

def cerca(paesaggio, aperto, compiti, rng, componenti=8):
    archivio = [np.zeros(componenti, dtype=int)]
    voti, figli = [voto(rng, paesaggio(archivio[0]), compiti)], [0]
    for _ in range(GIRI):
        if aperto:             # l'archivio: ogni voce può fare da genitore
            v = np.array(voti)
            peso = 1 / (1 + np.exp(-10 * (v - 0.5))) / (1 + np.array(figli))
            i = rng.choice(len(voti), p=peso / peso.sum())
        else:                  # il cricchetto: sempre dal migliore
            i = int(np.argmax(voti))
        figlio = archivio[i].copy()
        figlio[rng.integers(componenti)] ^= 1
        figli[i] += 1
        archivio.append(figlio)
        voti.append(voto(rng, paesaggio(figlio), compiti))
        figli.append(0)
    scelto = archivio[int(np.argmax(voti))]
    vero = paesaggio(scelto)
    return vero, max(voti), voto(rng, vero, 500), scelto[:3].sum() == 3

def medie(paesaggio, aperto, compiti, rng, componenti=8):
    r = [cerca(paesaggio, aperto, compiti, rng, componenti)
         for _ in range(PROVE)]
    return np.mean(r, axis=0)

rng = np.random.default_rng(0)
print("misura esatta, successo vero dell'harness scelto:")
for p, n in [(liscia, 8), (trappola, 8), (trappola, 16)]:
    c, a = medie(p, False, None, rng, n), medie(p, True, None, rng, n)
    riga = (f"  {p.__name__}, {n} componenti: "
            f"cricchetto {c[0]:.3f}, archivio {a[0]:.3f}")
    if p is trappola:
        riga += f"; blocco trovato {c[3]:.0%} e {a[3]:.0%}"
    print(riga)
for compiti in (50, 200):
    print(f"voto su {compiti} compiti (vero / voto con cui ha vinto / "
          f"voto su 500 nuovi):")
    for p in ((liscia, trappola) if compiti == 50 else (liscia,)):
        c, a = medie(p, False, compiti, rng), medie(p, True, compiti, rng)
        print(f"  {p.__name__}: cricchetto {c[0]:.3f} / {c[1]:.3f} / {c[2]:.3f}, "
              f"archivio {a[0]:.3f} / {a[1]:.3f} / {a[2]:.3f}")
tutti = np.ones(8, dtype=int)
print(f"massimo vero con 8 componenti: liscia {liscia(tutti):.2f}, "
      f"trappola {trappola(tutti):.2f}")